In [ ]:
# default_exp paper.plsr.learning_curve

# 3.1. Learning Curve (PLSR)

> Model performance as the number of training samples increases

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# mirzai utilities
from mirzai.data.loading import load_kssl
from mirzai.data.selection import (select_y, select_tax_order, select_X)
from mirzai.data.transform import (log_transform_y, SNV, TakeDerivative,
                                   DropSpectralRegions, CO2_REGION)

from fastcore.transform import compose

# Python utilities
from collections import OrderedDict
from tqdm.auto import tqdm
from pathlib import Path

# Data science stack
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.cross_decomposition import PLSRegression
from sklearn.metrics import mean_squared_error

import warnings
warnings.filterwarnings('ignore')

## Load and transform

In [ ]:
src_dir = '../_data'
fnames = ['spectra-features.npy', 'spectra-wavenumbers.npy', 
          'depth-order.npy', 'target.npy', 
          'tax-order-lu.pkl', 'spectra-id.npy']

X, X_names, depth_order, y, tax_lookup, X_id = load_kssl(src_dir, fnames=fnames)

data = X, y, X_id, depth_order

transforms = [select_y, select_tax_order, select_X, log_transform_y]
X, y, X_id, depth_order = compose(*transforms)(data)

In [ ]:
print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')
print(f'Wavenumbers:\n {X_names}')
print(f'depth_order (first 3 rows):\n {depth_order[:3, :]}')
print(f'Taxonomic order lookup:\n {tax_lookup}')

X shape: (40132, 1764)
y shape: (40132,)
Wavenumbers:
 [3999 3997 3995 ...  603  601  599]
depth_order (first 3 rows):
 [[43.  2.]
 [ 0.  0.]
 [ 0.  1.]]
Taxonomic order lookup:
 {'alfisols': 0, 'mollisols': 1, 'inceptisols': 2, 'entisols': 3, 'spodosols': 4, 'undefined': 5, 'ultisols': 6, 'andisols': 7, 'histosols': 8, 'oxisols': 9, 'vertisols': 10, 'aridisols': 11, 'gelisols': 12}


## Experiment

### Setup

In [ ]:
# Metrics to gather
history = OrderedDict({'nb_samples': [], 'train_score': [], 'valid_score': [], 'nb_components': []})

# Dataset subset sizes
training_size = [ 100, 500, 1000, 2500, 5000, 10000, 20000, 30000, X.shape[0]]

# Mean Square Error to target
target_mse = 0.01

### Run

In [ ]:
for size in tqdm(training_size):
    print(100*'-')
    print('# of samples: {}'.format(size))

    idx = np.random.choice(len(X), size, replace=False)
    X_train, X_valid, y_train, y_valid = train_test_split(X[idx, :], 
                                                          y[idx], 
                                                          test_size=0.2, 
                                                          random_state=42)

    if size < 2500:
        pls_range = range(2, 500, 10)
    else:
        pls_range = range(200, 1600, 200)

    early_stop = 1e-4
    last_score = 999

    for cpts in pls_range:
        pipe = Pipeline([('snv', SNV()),
                         ('derivative', TakeDerivative(window_length=11, polyorder=1)), 
                         ('dropper', DropSpectralRegions(X_names, regions=CO2_REGION)),
                         ('model', PLSRegression(n_components=cpts))])

        pipe.fit(X_train, y_train)
    
        train_score = mean_squared_error(pipe.predict(X_train), y_train)
    
        print("# of pls cpts: ", cpts)
        print("Train score: ", train_score)
        print("Last progress: ", last_score - train_score)
        if last_score - train_score <= early_stop:
            print("Early stopping!")
            break
        
        if train_score <= target_mse:
            print("Reached the target!")
            break

        last_score = train_score
    
    valid_score = mean_squared_error(pipe.predict(X_valid), y_valid)
    
    # Store scores and others in history
    print("Valid_score: ", valid_score)
    history['nb_samples'].append(len(X_train))
    history['train_score'].append(train_score)
    history['valid_score'].append(valid_score)
    history['nb_components'].append(cpts)

  0%|          | 0/9 [00:00<?, ?it/s]

----------------------------------------------------------------------------------------------------
# of samples: 100
# of pls cpts:  2
Train score:  0.09641579424646127
Last progress:  998.9035842057535
# of pls cpts:  12
Train score:  0.01671707520272305
Last progress:  0.07969871904373821
# of pls cpts:  22
Train score:  0.003427171916079294
Last progress:  0.013289903286643758
Reached the target!
Valid_score:  0.11168991750611439
----------------------------------------------------------------------------------------------------
# of samples: 500
# of pls cpts:  2
Train score:  0.09198502650154534
Last progress:  998.9080149734984
# of pls cpts:  12
Train score:  0.05148709687211881
Last progress:  0.040497929629426534
# of pls cpts:  22
Train score:  0.03592833155644815
Last progress:  0.015558765315670658
# of pls cpts:  32
Train score:  0.0264604941740063
Last progress:  0.009467837382441852
# of pls cpts:  42
Train score:  0.020383778145009517
Last progress:  0.006076716028996

### Save results

In [ ]:
dest_dir = 'name_of_your_destination_directory'

with open(Path(dest_dir)/'history_pls_learning_curve.pickle', 'wb') as f: 
    pickle.dump(history, f)